# 第 6 周解答：资助声明优先级分流（Grant Funding Statement Triage）

## 练习目标

把数据集 **cometadata/synthetic-funding-statements** 里的资助声明，按元数据完整度分成 **High / Medium / Low** 三类优先级。

## 流程（Flow）

1. 加载数据并派生标签  
2. Zero-shot 基线（OpenRouter）  
3. 准备 JSONL  
4. OpenAI Fine-tune  
5. 评估微调模型准确率  

## 和本课 Week 6 的关系

| 概念 | 本练习 |
|------|--------|
| 标签派生 | `derive_priority` 按 funder/award 字段计分 |
| 零样本基线 | OpenRouter + `gpt-4o-mini` |
| 微调数据 | Chat messages JSONL |
| 微调 API | `openai_ft.fine_tuning.jobs.create` |


In [ ]:
# ========== 导入：环境、数据、OpenAI、评价指标 ==========

# os：读环境变量（HF_TOKEN / OPENROUTER_API_KEY / OPENAI_API_KEY）
import os
# random：固定种子后打乱数据，保证可复现划分
import random
# json：序列化微调用的 JSONL 行
import json
# load_dotenv：从 .env 加载密钥
from dotenv import load_dotenv
# login：登录 Hugging Face Hub（拉数据集可能需要）
from huggingface_hub import login
# OpenAI：同时用于 OpenRouter 推理与 OpenAI 微调
from openai import OpenAI
# load_dataset：从 Hugging Face datasets 拉资助声明数据
from datasets import load_dataset
# accuracy_score：计算微调后样本准确率
from sklearn.metrics import accuracy_score


In [ ]:
# ========== 环境：加载密钥并做「是否已设置」的脱敏打印 ==========

# 加载 .env；override=True 覆盖已有同名环境变量
load_dotenv(override=True)
# Hugging Face token（必需键；缺失会 KeyError）
hf_token = os.environ["HF_TOKEN"]
# 登录 HF，便于后续访问 Hub 资源
login(hf_token, add_to_git_credential=True)

# OpenRouter：零样本基线推理；OpenAI：微调（两者密钥不同）
openrouter_api_key = os.environ.get("OPENROUTER_API_KEY")
openai_api_key = os.environ.get("OPENAI_API_KEY")

# 只打印密钥末几位，确认已加载且不泄露全文
if openrouter_api_key:
    print(f"OpenRouter API Key: ...{openrouter_api_key[-3:]}")
else:
    print("OPENROUTER_API_KEY not set")
if openai_api_key:
    print(f"OpenAI API Key: ...{openai_api_key[-4:]}")
else:
    print("OPENAI_API_KEY not set (needed for fine-tuning)")


In [4]:
# ========== 标签空间 + 从元数据完整度派生 High/Medium/Low ==========

# 允许的优先级标签（顺序用于后面子串匹配）
PRIORITIES = ("High", "Medium", "Low")

# 根据 funders/awards 字段是否齐全打分，映射到三级优先级
def derive_priority(row):
    # 根据资助元数据完整度派生优先级：High=完整，Low=稀疏。
    # 累计「有信息」的字段数
    score = 0
    # funders 可能为 None：用空列表兜底
    funders = row.get("funders") or []
    # 没有任何 funder → 直接 Low
    if not funders:
        return "Low"
    # 只看第一个 funder（与原逻辑一致）
    f = funders[0]
    # 有资助方名称 +1
    if f.get("funder_name"):
        score += 1
    # awards 也可能为空
    awards = f.get("awards") or []
    if awards:
        a = awards[0]
        # 有非空 award_ids +1
        if a.get("award_ids") and len(a["award_ids"]) > 0:
            score += 1
        # 有非空 award_title +1
        if a.get("award_title") and len(a["award_title"]) > 0:
            score += 1
        # 有非空 funding_scheme +1
        if a.get("funding_scheme") and len(a["funding_scheme"]) > 0:
            score += 1
    # 阈值：>=3 High，=2 Medium，否则 Low
    if score >= 3:
        return "High"
    if score == 2:
        return "Medium"
    return "Low"


In [ ]:
# ========== 加载数据集、打标签、70/15/15 划分 ==========

# 从 Hub 拉取合成资助声明训练集
ds = load_dataset("cometadata/synthetic-funding-statements", split="train")
# 每行保留声明文本 + 派生出的 priority 标签
examples = [
    {"funding_statement": row["funding_statement"], "priority": derive_priority(row)}
    for row in ds
]

# 固定种子保证划分可复现
random.seed(42)
shuffled = examples.copy()
random.shuffle(shuffled)
n = len(shuffled)
# 70% train / 15% val / 剩余 test
n_train, n_val = int(0.7 * n), int(0.15 * n)
train_data = shuffled[:n_train]
val_data = shuffled[n_train : n_train + n_val]
test_data = shuffled[n_train + n_val :]

# 打印各子集规模
print(f"Loaded {len(train_data):,} train, {len(val_data):,} val, {len(test_data):,} test")


In [6]:
# ========== 客户端：OpenRouter 推理 + OpenAI 微调 ==========

# OpenRouter 兼容 OpenAI SDK：换 base_url 即可
openrouter = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=openrouter_api_key)

# 没有 OPENAI_API_KEY 时 openai_ft 为 None，后面单元格会跳过上传/微调
openai_ft = OpenAI(api_key=openai_api_key) if openai_api_key else None
# 零样本基线使用的前沿小模型（OpenRouter 路由名，保持原样）
FRONTIER_MODEL = "openai/gpt-4o-mini"


In [7]:
# ========== 控制微调成本：截取 train/val 子集 ==========

# 微调不必用全量：500 条训练 + 100 条验证通常够演示
fine_tune_train = train_data[:500]
fine_tune_validation = val_data[:100]
print(f"Fine-tune: {len(fine_tune_train)} train, {len(fine_tune_validation)} val")


Fine-tune: 500 train, 100 val


In [8]:
# ========== 快速确认微调训练集长度 ==========

# 交互检查：应等于上一格截取后的长度（500，若数据够）
len(fine_tune_train)


500

In [10]:
# ========== 零样本基线：让前沿模型只回复 High/Medium/Low ==========

# 对单条资助声明做 triage；无 openrouter 时退回 Medium
def predict_baseline(statement: str) -> str:
    # 客户端未配置时给保守默认，避免整条流水线崩掉
    if not openrouter:
        return "Medium"
    # system prompt 规定标签定义与输出格式（英文原样，禁止改写）
    sys_prompt = (
        "You triage grant funding statements. Classify each as exactly one of: High, Medium, Low. "
        "High = complete (funder, award ID, scheme). Medium = partial. Low = sparse. Reply with only that word."
    )
    # Chat Completions：短输出即可（max_tokens=10）
    r = openrouter.chat.completions.create(
        model=FRONTIER_MODEL,
        messages=[
            {"role": "system", "content": sys_prompt},
            {"role": "user", "content": statement},
        ],
        max_tokens=10,
    )
    # 取出模型原文并去空白
    raw = (r.choices[0].message.content).strip()
    # 在回复里子串匹配合法标签（大小写不敏感）
    for label in PRIORITIES:
        if label.lower() in raw.lower():
            return label
    # 匹配失败则返回原文，或再退回 Medium
    return raw or "Medium"


In [11]:
# ========== 通用准确率辅助：预测标签 == 真值 的比例 ==========

# predictor 接收 funding_statement 字符串，返回优先级标签
def accuracy(predictor, data):
    # 逐条比对；空数据返回 0.0 避免除零
    correct = sum(1 for ex in data if predictor(ex["funding_statement"]) == ex["priority"])
    return correct / len(data) if data else 0.0


In [ ]:
# ========== 基线评估：先在 50 条测试样本上看零样本准确率 ==========

# 用 API 调用，样本数刻意缩小以控制费用与耗时
baseline_acc = accuracy(predict_baseline, test_data[:50])
print(f"Zero-shot baseline accuracy (sample): {baseline_acc:.1%}")


# 步骤 1 — 准备 JSONL 并上传到 OpenAI

把训练数据写成 **JSONL（JSON Lines）** 格式：每行一条 `{"messages": [...]}`，再上传给 OpenAI Fine-tuning API。


In [13]:
# ========== 构造单条微调样本的 messages ==========

# user = 资助声明文本；assistant = 目标优先级标签
def messages_for(ex):
    # 一条训练样本：user=资助声明，assistant=优先级。
    return [
        {"role": "user", "content": ex["funding_statement"]},
        {"role": "assistant", "content": ex["priority"]},
    ]


In [ ]:
# ========== 抽查：看第一条微调样本长什么样 ==========

# 应看到 user/assistant 两条 message
messages_for(fine_tune_train[0])


In [15]:
# ========== 把样本列表变成 JSONL 字符串 ==========

# 每行一个 JSON 对象，字段 messages 为 chat 对话
def make_jsonl(items):
    return "\n".join(json.dumps({"messages": messages_for(ex)}) for ex in items)


In [16]:
# ========== 写 JSONL 到磁盘 ==========

# filename 由调用方指定（train / validation 各一份）
def write_jsonl(items, filename):
    with open(filename, "w") as f:
        jsonl = make_jsonl(items)
        f.write(jsonl)


In [17]:
# ========== 写出训练集 JSONL ==========

# 确保目录存在，再写入 fine_tune_train
os.makedirs("jsonl", exist_ok=True)
write_jsonl(fine_tune_train, "jsonl/fine_tune_train.jsonl")


In [18]:
# ========== 写出验证集 JSONL ==========

# 验证文件供微调作业监控过拟合 / 选 checkpoint
write_jsonl(fine_tune_validation, "jsonl/fine_tune_validation.jsonl")


In [ ]:
# ========== 上传训练文件到 OpenAI Files API ==========

# 成功后拿到 file id，供 fine_tuning.jobs.create 使用
train_file = None
if openai_ft:
    # purpose 必须是 fine-tune
    with open("jsonl/fine_tune_train.jsonl", "rb") as f:
        train_file = openai_ft.files.create(file=f, purpose="fine-tune")
    print(f"Uploaded train file: {train_file.id}")
else:
    print("Skipping upload: OPENAI_API_KEY not set")


In [ ]:
# ========== 查看上传后的 train_file 对象 ==========

# 交互确认：应含 id / bytes / purpose 等字段
train_file


In [ ]:
# ========== 上传验证文件 ==========

validation_file = None
if openai_ft:
    with open("jsonl/fine_tune_validation.jsonl", "rb") as f:
        validation_file = openai_ft.files.create(file=f, purpose="fine-tune")
    print(f"Uploaded validation file: {validation_file.id}")


In [ ]:
# ========== 查看 validation_file 对象 ==========

validation_file


## 文件控制台

上传后可在 OpenAI 存储页查看文件：  
https://platform.openai.com/storage/files/


# 步骤 2 — 启动微调（Fine-tune）

用已上传的 train / validation 文件创建微调作业。作业在云端异步跑，需稍后轮询状态。


In [ ]:
# ========== 创建微调作业：gpt-4.1-nano + 小超参 ==========

job_id = None
# 三者齐全才创建作业，避免半成品调用报错
if openai_ft and train_file and validation_file:
    job = openai_ft.fine_tuning.jobs.create(
        training_file=train_file.id,
        validation_file=validation_file.id,
        # 基础模型 id 保持原样
        model="gpt-4.1-nano-2025-04-14",
        seed=42,
        # 1 epoch、batch_size=1：省钱的演示配置
        hyperparameters={"n_epochs": 1, "batch_size": 1},
        # 后缀会出现在微调模型名里，便于识别
        suffix="grant-triage",
    )
    job_id = job.id
    print(f"Fine-tuning job: {job_id}")
else:
    print("Skipping fine-tune: run upload cells first")


In [25]:
# ========== 列出最近微调作业（快速扫一眼队列） ==========

if openai_ft:
    openai_ft.fine_tuning.jobs.list(limit=5)


In [ ]:
# ========== 若本会话没记下 job_id，则取最近一条作业 ==========

# 重新打开笔记本时常用：从 list 取最新 job
if job_id is None and openai_ft:
    job_id = openai_ft.fine_tuning.jobs.list(limit=1).data[0].id
job_id


In [ ]:
# ========== 再次确认当前 job_id ==========

job_id


In [ ]:
# ========== 检索作业详情：看 status / fine_tuned_model ==========

# succeeded 后会带上 fine_tuned_model 字段
openai_ft.fine_tuning.jobs.retrieve(job_id) if openai_ft and job_id else None


In [ ]:
# ========== 拉取作业事件日志（最近 10 条） ==========

# 可看到 training 进度、报错信息等
openai_ft.fine_tuning.jobs.list_events(fine_tuning_job_id=job_id, limit=10).data if openai_ft and job_id else []


## 微调控制台

也可在网页查看作业进度：  
https://platform.openai.com/finetune


# 步骤 3 — 测试微调后的模型

作业成功后，用 `fine_tuned_model` 在测试集上做预测，并与真值对比。


In [39]:
# ========== 从已完成作业取出微调模型名 ==========

# 形如 ft:gpt-4.1-nano-...:grant-triage:...
fine_tuned_model_name = openai_ft.fine_tuning.jobs.retrieve(job_id).fine_tuned_model


In [ ]:
# ========== 查看微调模型 id ==========

fine_tuned_model_name


In [41]:
# ========== 推理用 messages：只要 user（不要泄露标签） ==========

def test_messages_for(ex):
    return [{"role": "user", "content": ex["funding_statement"]}]


In [ ]:
# ========== 抽查测试消息格式 ==========

test_messages_for(test_data[0])


In [43]:
# ========== 微调模型预测：同样规范到 High/Medium/Low ==========

def predict_finetuned(ex):
    # 微调模型预测优先级（需要已得到 fine_tuned_model_name）。
    # 模型名或客户端缺失时退回 Medium
    if not fine_tuned_model_name or not openai_ft:
        return "Medium"
    # 走 OpenAI Chat Completions（微调模型不能用 OpenRouter）
    response = openai_ft.chat.completions.create(
        model=fine_tuned_model_name,
        messages=test_messages_for(ex),
        max_tokens=10,
    )

    # 规范化输出到 PRIORITIES 之一
    raw = (response.choices[0].message.content or "").strip()
    for label in PRIORITIES:
        if label.lower() in raw.lower():
            return label
    return raw or "Medium"


In [ ]:
# ========== 定性抽查：打印前 5 条真值 vs 预测 ==========

for ex in test_data[:5]:
    pred = predict_finetuned(ex)
    # 只截断声明前 60 字符，避免刷屏
    print(f"True: {ex['priority']} | Pred: {pred} | {ex['funding_statement'][:60]}...")


In [ ]:
# ========== 定量评估：前 100 条测试样本准确率 ==========

if fine_tuned_model_name:
    # 批量预测与对齐真值
    preds = [predict_finetuned(ex) for ex in test_data[:100]]
    truths = [ex["priority"] for ex in test_data[:100]]
    # sklearn accuracy_score：一致样本占比
    acc = accuracy_score(truths, preds)
    print(f"Fine-tuned accuracy (sample): {acc:.1%}")
else:
    print("Run fine-tuning first to evaluate")
